# Aligning timestamps to label window

### Aim

This notebook documents progress on a function that aligns article publication timestamps to our label windows, noting edge cases and conventions along the way. It also saves the schedule we will use for the rest of the project, for calculating features and labels.

**Output:** `schedule.parquet`

### Importance

The code for this function is important to the project as it is the leakage frontier: if an article published *after* a price move gets labeled with the move it's reporting on, every downstream result becomes impressive-looking fiction, and no error will ever be thrown. We need to make sure that we are following strict conventions and accounting for any holidays or unusual market closures.

### Convention

Here are the conventions we use for the function. There are two possible timestamp types we can encounter: *market hours* and *after market hours*. Since we work with the NYSE, timestamps must map to the correct timezone. We work in **UTC**, so the label windows must correspond to this as well.

A regular NYSE trading day opens at 09:30 ET and closes at 16:00 ET, and is open Monday to Friday. Since our timestamps are in UTC, we need the times to match this timezone. Because ET shifts clocks twice a year, the regular market times in UTC change depending on the time of year:
- March to October -> 13:30 UTC to 20:00 UTC
- October to March -> 14:30 UTC to 21:00 UTC

The bottom line: every timestamp should correspond to the **next full trading day**, so we can start the label window from there. Some examples follow (assuming we are in August).
 
- During market hours -> next market opening
    - E.g. Friday 19:00 UTC -> Monday 13:30 UTC 
    - E.g. Wednesday 19:59 UTC -> Thursday 13:30 UTC
- After market hours -> next market opening
    - E.g. Sunday 13:00 UTC -> Monday 13:30 UTC
    - E.g. Tuesday 13:29 UTC -> Tuesday 13:30 UTC
- Abnormal holidays -> next market opening
    - E.g. 1st January 12:00 UTC (New Years Day Closure) -> 2nd January 14:30 UTC

### Pulling Market Calendar

First, we pull the real market calendar data for this function using `pandas_market_calendars`. As with pulling OHLCV data, we need a wider range than our publication date range, since we need to look forward 3 full trading days for the abnormal return label.

Looking forward by 10 days is enough to account for weekends and holidays, with a small extra buffer afterward. We also extend the lower end by one day to account for articles published the morning of the timeframe's start, before trading begins, since we still want these accepted.

In [1]:
import pandas_market_calendars as mcal
import pandas as pd
import datetime as dt
from stock_predictor.config import NEWS_START_DATE, NEWS_END_DATE, RAW_DATA_DIR

schedule_start = pd.Timestamp(NEWS_START_DATE) - dt.timedelta(days=1)
schedule_end = pd.Timestamp(NEWS_END_DATE) + dt.timedelta(days=10)

print(f"Schedule start date: {schedule_start}")
print(f"Schedule end date: {schedule_end}")

nyse = mcal.get_calendar("NYSE")

schedule = nyse.schedule(start_date=schedule_start, end_date=schedule_end)
print(schedule)
print(schedule.dtypes)

2026-08-26 23:34:13.075 | INFO     | stock_predictor.config:<module>:12 - PROJ_ROOT path is: C:\Users\kacpe\OneDrive - University of Warwick\PROJECTS\Stock Predictor\stock-predictor


Schedule start date: 2025-07-31 00:00:00
Schedule end date: 2026-08-11 00:00:00
                         market_open              market_close
2025-07-31 2025-07-31 13:30:00+00:00 2025-07-31 20:00:00+00:00
2025-08-01 2025-08-01 13:30:00+00:00 2025-08-01 20:00:00+00:00
2025-08-04 2025-08-04 13:30:00+00:00 2025-08-04 20:00:00+00:00
2025-08-05 2025-08-05 13:30:00+00:00 2025-08-05 20:00:00+00:00
2025-08-06 2025-08-06 13:30:00+00:00 2025-08-06 20:00:00+00:00
...                              ...                       ...
2026-08-05 2026-08-05 13:30:00+00:00 2026-08-05 20:00:00+00:00
2026-08-06 2026-08-06 13:30:00+00:00 2026-08-06 20:00:00+00:00
2026-08-07 2026-08-07 13:30:00+00:00 2026-08-07 20:00:00+00:00
2026-08-10 2026-08-10 13:30:00+00:00 2026-08-10 20:00:00+00:00
2026-08-11 2026-08-11 13:30:00+00:00 2026-08-11 20:00:00+00:00

[259 rows x 2 columns]
market_open     datetime64[us, UTC]
market_close    datetime64[us, UTC]
dtype: object


The datetimes are in UTC, as needed. This can now be saved as a parquet file for future use.

In [2]:
schedule.to_parquet(RAW_DATA_DIR / "raw_schedule.parquet")

Let's check for holidays so we can use them as an edge case when testing the function, by checking for every weekday not included in our list of trading days.

In [3]:
# Get all weekdays
all_days = pd.date_range(start=NEWS_START_DATE, end=schedule_end, freq="D")
weekdays = all_days[all_days.dayofweek < 5]

# Normalise trading days
trading_days = schedule.index.normalize()

# Filter to show holidays i.e. non trading days
holidays = weekdays[~weekdays.normalize().isin(trading_days)].normalize()

print(holidays)

DatetimeIndex(['2025-09-01', '2025-11-27', '2025-12-25', '2026-01-01',
               '2026-01-19', '2026-02-16', '2026-04-03', '2026-05-25',
               '2026-06-19', '2026-07-03'],
              dtype='datetime64[us]', freq=None)


We can now use this information to write our function and tests.

### Showcasing examples

Let's apply the function to the examples from our convention to show it works. We write a helper that makes timezone-aware datetime objects, so we can easily pass in datetimes as they would appear in article timestamps; this is needed so pandas can compare the timestamps correctly.

In [4]:
from stock_predictor.market.timestamp_alignment import align_timestamp, make_dt

example_cases = [
    {"label": "Friday market hours",     "date": "2026-07-03", "time": "19:00:00"},
    {"label": "Wednesday just before close", "date": "2026-07-01", "time": "19:59:00"},
    {"label": "Sunday after-hours",      "date": "2026-07-05", "time": "13:00:00"},
    {"label": "Tuesday just before open","date": "2026-07-07", "time": "13:29:00"},
    {"label": "Christmas Day",           "date": "2026-01-01", "time": "12:00:00"},
]

for case in example_cases:
    pub_dt = make_dt(case["date"], case["time"])
    result = align_timestamp(pub_dt, schedule)
    print(f"{case['label']:35s} | {pub_dt} -> {result}")

Friday market hours                 | 2026-07-03 19:00:00+00:00 -> 2026-07-06 13:30:00+00:00
Wednesday just before close         | 2026-07-01 19:59:00+00:00 -> 2026-07-02 13:30:00+00:00
Sunday after-hours                  | 2026-07-05 13:00:00+00:00 -> 2026-07-06 13:30:00+00:00
Tuesday just before open            | 2026-07-07 13:29:00+00:00 -> 2026-07-07 13:30:00+00:00
Christmas Day                       | 2026-01-01 12:00:00+00:00 -> 2026-01-02 14:30:00+00:00


The function behaves as intended in these examples. It will also be tested more formally with thorough unit tests covering all edge cases.

### Note

The function takes the NYSE schedule as an argument, along with the publication timestamp. This avoids pulling a new schedule for each call, since we know all articles sit within our timeframe: we can pull the schedule once and reuse it for each timestamp.

The function also checks that the publication timestamp is within the timeframe of the schedule. This works because, as earlier in the notebook, we pull a schedule based on the timeframe the articles are drawn from (with some leeway added), so no article should fall outside it. There is also a separate check that the publication date itself is within our timeframe, which raises an error even if the schedule happens to be correct.

As mentioned earlier, articles published in the morning of the timeframe's start are still accepted, with the label window beginning the same day.